# Skyfusion Analytics · Modelos v1 (Avance 2)
### Minería de Datos · UNIMINUTO Ibagué · 2026-2
**Contexto:** Monitoreo Ambiental y Gestión del Riesgo en la Cuenca del Río Combeima  
**Entregable:** Consolidado de Modelos Supervisados (Árbol, Naive Bayes, KNN) y No Supervisados (K-Means, Jerárquico)

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, classification_report, silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline


## 1. Carga de Datos y Split Congelado GEE
Retomamos el dataset oficial de Sentinel-2 con 1.924 filas y el split congelado estratificado (1.443 train / 481 test, semilla 42).

In [ ]:
base_dir = Path('../')
data_path = base_dir / 'sesion7' / 'datasets' / 'sentinel2_combeima_clases.csv'
split_path = base_dir / 'sesion7' / 'resultados' / 'metricas_split.json'

df = pd.read_csv(data_path)
with open(split_path, 'r', encoding='utf-8') as f:
    split_meta = json.load(f)

features = split_meta['features_orden']
X = df[features].astype(float)
y = df['clase'].astype(str)

train_idx = split_meta['indices_train']
test_idx = split_meta['indices_test']

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print(f'Dataset cargado: {len(df)} filas, {len(features)} features')
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
display(df.head())


## 2. Parte I: Torneo de Clasificación Supervisada
Competencia directa: Árbol depth 5 (entropía) vs GaussianNB vs KNN (k=1, 5, 11, 31, 101).

In [ ]:
modelos = {
    'Árbol depth 5 (entropía)': DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42),
    'GaussianNB': GaussianNB(),
    'KNN k=5 (sin escalar)': KNeighborsClassifier(n_neighbors=5),
    'KNN k=1 (escalado)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1)),
    'KNN k=5 (escalado)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    'KNN k=11 (escalado)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=11)),
    'KNN k=31 (escalado)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=31)),
    'KNN k=101 (escalado)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=101)),
}

resultados = []
for nombre, mod in modelos.items():
    mod.fit(X_train, y_train)
    y_tr = mod.predict(X_train)
    y_te = mod.predict(X_test)
    resultados.append({
        'Modelo': nombre,
        'Accuracy Train': round(accuracy_score(y_train, y_tr), 4),
        'Accuracy Test': round(accuracy_score(y_test, y_te), 4),
        'Macro-F1 Test': round(f1_score(y_test, y_te, average='macro'), 4),
        'Balanced Acc Test': round(balanced_accuracy_score(y_test, y_te), 4)
    })

res_df = pd.DataFrame(resultados)
display(res_df)


## 3. Parte II: Aprendizaje No Supervisado (K-Means y Jerárquico)
Descubrimiento de agrupaciones espectrales naturales en la cuenca sin supervisión de etiquetas.

In [ ]:
bandas = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
X_bandas = df[bandas].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_bandas)

# Curvas de Codo e Inercia
ks = list(range(2, 9))
inercias = []
siluetas = []

for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_scaled, labels))

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()
ax1.plot(ks, inercias, 'bo-', label='Inercia (Codo)')
ax2.plot(ks, siluetas, 'r^-', label='Coeficiente de Silueta')
ax1.set_xlabel('Número de clusters (k)')
ax1.set_ylabel('Inercia', color='b')
ax2.set_ylabel('Silueta', color='r')
plt.title('Búsqueda de k óptimo: Inercia y Silueta para la Cuenca del Combeima')
plt.show()


### Segmentación Final k=3 y Perfilamiento Espectral

In [ ]:
km_final = KMeans(n_clusters=3, n_init=10, random_state=42)
df['cluster'] = km_final.fit_predict(X_scaled)

perfiles = df.groupby('cluster')[bandas].mean().round(1)
composicion_wc = (df.groupby('cluster')['clase'].value_counts(normalize=True)*100).round(1).unstack(fill_value=0)

display(pd.concat([df['cluster'].value_counts().rename('Pixeles'), perfiles, composicion_wc], axis=1))


### Dendrograma Jerárquico (Ward) sobre Muestra de 60 Píxeles

In [ ]:
sample_indices = df.groupby('cluster', group_keys=False).apply(lambda s: s.sample(min(len(s), 20), random_state=42)).index
sample_X = X_scaled[sample_indices]
sample_labels = df.loc[sample_indices, 'cluster'].values

Z = linkage(sample_X, method='ward')

plt.figure(figsize=(12, 5))
dendrogram(Z, labels=[f'C{c}' for c in sample_labels], color_threshold=6.5)
plt.axhline(y=6.5, color='crimson', linestyle='--', label='Corte k=3')
plt.title('Dendrograma de Clustering Jerárquico (Ward) - Muestra 60 Píxeles')
plt.ylabel('Distancia de Fusión')
plt.legend()
plt.show()
